In [1]:
import pandas as pd
import numpy as np
np.random.seed(42)
path = r"C:\univ\Data mining\Project\jsons\all_final_appended.json"

df = pd.read_json(path)

print(df.head())
print(df.info())

    business_name   sector  followers_count   post_date posting_hour  \
0  LOFT Palestine  Fashion           4392.0  2026-03-31         None   
1  LOFT Palestine  Fashion           4392.0  2026-03-18         None   
2  LOFT Palestine  Fashion           4392.0  2026-03-16         None   
3  LOFT Palestine  Fashion           4392.0  2026-03-07         None   
4  LOFT Palestine  Fashion           4392.0  2026-03-05         None   

  day_of_week  month post_type  \
0     Tuesday  March      reel   
1   Wednesday  March      reel   
2      Monday  March      reel   
3    Saturday  March      reel   
4    Thursday  March      reel   

                                        caption_text  caption_length  ...  \
0  إطلالة أنثوية بلمسة عصرية ✨ من LOFT. للطلب الت...              88  ...   
1  اختيارك المثالي لإطلالة كاجول مميزة 🤍✨ LOFT كو...             111  ...   
2  ستايل شبابي بلمسة عصرية له ولها ✨ اختيارات ممي...             133  ...   
3  ستايل يناسبك، وتجربة تستحق الزيارة ✨ زورونا في...  

In [2]:
df.isna().sum()/len(df)

business_name           0.000000
sector                  0.000000
followers_count         0.060563
post_date               0.026761
posting_hour            0.980282
day_of_week             0.074648
month                   0.047887
post_type               0.000000
caption_text            0.000000
caption_length          0.000000
hashtags_count          0.000000
emoji_count             0.000000
likes_count             0.070423
comments_count          0.150704
views_count             0.281690
language                0.000000
CTA_present             0.000000
promo_post              0.000000
discount_percent        0.928169
mentions_location       0.000000
religious_theme         0.000000
patriotic_theme         0.000000
arabic_dialect_style    0.000000
sponsored               0.056338
kind                    0.998592
dtype: float64

In [3]:
df = df.drop(columns=['kind', 'posting_hour', 'discount_percent'])

In [4]:
def print_missing():
    missing_ratio = df.isna().sum() / len(df)

    # Get columns where missing percentage > 0
    missing_columns = missing_ratio[missing_ratio > 0]

    print(missing_columns)

In [5]:
print_missing()

followers_count    0.060563
post_date          0.026761
day_of_week        0.074648
month              0.047887
likes_count        0.070423
comments_count     0.150704
views_count        0.281690
sponsored          0.056338
dtype: float64


### `handle sponsored missing`

In [6]:
df.loc[:, 'sponsored'] = df['sponsored'].fillna(0)
df['sponsored'].value_counts()


sponsored
0.0    658
1.0     52
Name: count, dtype: int64

### `EDA`

In [7]:
# Keep only columns that have at least one missing value in the whole dataframe
missing_cols = df.columns[df.isna().any()]

# Percentage of missing values grouped by post_type
missing_by_post_type = (
    df.groupby("post_type")[missing_cols]
      .apply(lambda x: x.isna().sum() / len(x))
)

print(missing_by_post_type)

           followers_count  post_date  day_of_week     month  likes_count  \
post_type                                                                   
Image             0.090909   0.040404     0.040404  0.040404     0.080808   
Reel              0.047059   0.011765     0.011765  0.011765     0.023529   
post              0.000000   0.062500     0.125000  0.062500     0.187500   
reel              0.058824   0.025490     0.090196  0.054902     0.072549   

           comments_count  views_count  
post_type                               
Image            0.161616     0.959596  
Reel             0.117647     0.035294  
post             0.125000     1.000000  
reel             0.154902     0.168627  


In [8]:
df.columns

Index(['business_name', 'sector', 'followers_count', 'post_date',
       'day_of_week', 'month', 'post_type', 'caption_text', 'caption_length',
       'hashtags_count', 'emoji_count', 'likes_count', 'comments_count',
       'views_count', 'language', 'CTA_present', 'promo_post',
       'mentions_location', 'religious_theme', 'patriotic_theme',
       'arabic_dialect_style', 'sponsored'],
      dtype='object')

In [9]:
def print_inconsistencies():
       categ_cols = set(df.columns) - set(['business_name', 'followers_count', 'post_date','caption_text', 'caption_length',
              'hashtags_count', 'emoji_count', 'likes_count', 'comments_count',
              'views_count',])
       for col in categ_cols:
              print(f"{col}: {df[col].nunique()}")
              print(f"{col}: {df[col].unique()}")

In [10]:
print_inconsistencies()

sponsored: 2
sponsored: [0. 1.]
month: 12
month: ['March' 'February' None 'April' 'October' 'May' 'September' 'December'
 'August' 'June' 'January' 'July' 'November']
sector: 5
sector: ['Fashion' 'Cafes/Restaurants' 'Influencers' 'Gym' 'Supermarkets']
religious_theme: 2
religious_theme: [False  True]
patriotic_theme: 2
patriotic_theme: [False  True]
promo_post: 2
promo_post: [False  True]
day_of_week: 7
day_of_week: ['Tuesday' 'Wednesday' 'Monday' 'Saturday' 'Thursday' None 'Friday'
 'Sunday']
arabic_dialect_style: 3
arabic_dialect_style: [True False 'Levantine Arabic']
CTA_present: 2
CTA_present: [ True False]
post_type: 4
post_type: ['reel' 'post' 'Image' 'Reel']
language: 12
language: ['Arabic' 'English' 'none' 'mixed' 'None' 'Arabic-English Mixed' 'Mixed'
 'Arabic/English Mixed' 'Emoji Only' 'No Caption' 'English/Arabic Mixed'
 'Unknown']
mentions_location: 2
mentions_location: [ True False]


### `Handle Language inconsistency`

In [11]:
len(df[df['caption_length']==0]['language'])

20

In [12]:
len(df[df['language'].isin([ 'none',  'None' ,'No Caption', 'Unknown'])])

20

In [13]:
emoji_mask = df['language']=='Emoji Only'
no_caption_mask = df['language'].isin([ 'none',  'None' ,'No Caption', 'Unknown'])

In [14]:
df.loc[emoji_mask | no_caption_mask, 'language'] = 'No Caption'

In [15]:
df['language'].unique()

array(['Arabic', 'English', 'No Caption', 'mixed', 'Arabic-English Mixed',
       'Mixed', 'Arabic/English Mixed', 'English/Arabic Mixed'],
      dtype=object)

In [16]:
mixed_mask = df['language'].isin([
    'mixed',
    'Arabic-English Mixed',
    'Mixed',
    'Arabic/English Mixed',
    'English/Arabic Mixed'
])

df.loc[mixed_mask, 'language'] = 'Mixed'

In [17]:
df['language'].unique()

array(['Arabic', 'English', 'No Caption', 'Mixed'], dtype=object)

In [18]:
df['language'].isna().sum()

np.int64(0)

`Language inconsistency handled`

### `Date interpolation and missing handlgin`

In [19]:
df[df['day_of_week'].isna()]['business_name'].value_counts()

business_name
Metro Market Nablus                  16
Balcon Boutique & Cafe                9
Fashion strawberry                    9
Bravo Supermarket                     8
Pardo Cafe                            5
drift.for.men                         2
LOFT Palestine                        1
Hijjawi Fashion - الحجاوي للأزياء     1
528 Fashion                           1
Hulk Gym / Diaa Shabaro               1
Name: count, dtype: int64

In [20]:
df['post_date'].unique()

array(['2026-03-31', '2026-03-18', '2026-03-16', '2026-03-07',
       '2026-03-05', '2026-02-26', '2026-02-17', '2026-02-16', None,
       '19h ago', '1d ago', '2d ago', '3d ago', '4d ago', '5d ago',
       '1w ago', 'April 29', '28 April', '2025-10-22', '2025-03-07',
       '2026-05-07', '2026-05-06', '2026-05-05', '2026-05-04',
       '2026-05-03', '2026-05-02', '2025-02-23', '2024-09-29',
       '2024-12-23', '2026-04-30', '2026-04-23', '2026-03-24',
       '2026-03-14', '2025-08-10', '2025-05-21', '2026-04-28',
       '2026-04-25', '2026-04-24', '2023-10-03', '2023-09-28',
       '2023-09-26', '2023-09-24', '2023-06-03', '2023-05-27',
       '2023-05-17', '2023-05-07', '2026-04-29', '2026-04-22',
       '2026-04-13', '2026-04-17', '2026-05-01', '2026-04-11',
       '2026-04-04', '2026-03-15', '2026-03-13', '2025-05-05',
       '2025-05-07', '2026-03-17', '2026-03-19', '2026-03-20',
       '2026-03-21', '2026-03-22', '2026-03-23', '2026-03-27',
       '2025-05-06', '2025-05-04', '20

In [21]:
import re

filtered_dates = [
    x for x in df['post_date'].unique()
    if not (
        isinstance(x, str) and (
            # matches: 2025-05-07 or 2025/05/07
            re.fullmatch(r'\d{4}[-/]\d{2}[-/]\d{2}', x.strip())
            or
            # matches: April 30
            re.fullmatch(
                r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2}',
                x.strip()
            )
        )
    )
]

print(filtered_dates)

[None, '19h ago', '1d ago', '2d ago', '3d ago', '4d ago', '5d ago', '1w ago', '28 April', '14 hours ago', '2 days ago', '3 days ago', '4 days ago', '5 days ago', '6 days ago', '1 week ago', '20 hours ago']


In [22]:
import pandas as pd
import re

# reference date = 7 days ago
now = pd.Timestamp.now().normalize() - pd.Timedelta(days=7)

def parse_post_date(x):
    if pd.isna(x):
        return pd.NaT

    x = str(x).strip()

    # ---------- AGO FORMATS ----------
    ago_map = {
        '19h ago': pd.Timedelta(hours=19),
        '20 hours ago': pd.Timedelta(hours=20),
        '14 hours ago': pd.Timedelta(hours=14),

        '1d ago': pd.Timedelta(days=1),
        '2d ago': pd.Timedelta(days=2),
        '3d ago': pd.Timedelta(days=3),
        '4d ago': pd.Timedelta(days=4),
        '5d ago': pd.Timedelta(days=5),

        '2 days ago': pd.Timedelta(days=2),
        '3 days ago': pd.Timedelta(days=3),
        '4 days ago': pd.Timedelta(days=4),
        '5 days ago': pd.Timedelta(days=5),
        '6 days ago': pd.Timedelta(days=6),

        '1w ago': pd.Timedelta(weeks=1),
        '1 week ago': pd.Timedelta(weeks=1),
    }

    if x in ago_map:
        return now - ago_map[x]

    # ---------- PARSE NORMAL DATES ----------
    parsed = pd.to_datetime(x, errors='coerce')

    # if year missing, assume current year
    if pd.notna(parsed) and parsed.year == 1900:
        parsed = parsed.replace(year=now.year)

    return parsed


df['post_date'] = df['post_date'].apply(parse_post_date)

In [23]:
# show rows where the datetime is NOT midnight (00:00:00)
# i.e. values like 2026-05-05 05:00:00

print(len(df[df['post_date'].dt.time != pd.Timestamp('00:00:00').time()]))
print(len(df[df['post_date'].isna()]))
# show the non-missing rows that have a non-midnight time

df[
    df['post_date'].notna() &
    (df['post_date'].dt.time != pd.Timestamp('00:00:00').time())
]

36
33


,business_name,sector,followers_count,post_date,day_of_week,month,post_type,caption_text,caption_length,hashtags_count,...,comments_count,views_count,language,CTA_present,promo_post,mentions_location,religious_theme,patriotic_theme,arabic_dialect_style,sponsored
9,Balcon Boutique & Cafe,Fashion,243000.0,2026-05-05 05:00:00,None,None,reel,,0,0,...,NaN,13400.0,No Caption,False,True,False,False,False,False,0.0
670,Bravo Supermarket,Supermarkets,18000.0,2026-05-05 10:00:00,None,None,reel,من أكثر من 25 سنة، وبرافو بتختار فروعها بعناية...,323,0,...,NaN,4390.0,Arabic,True,True,True,False,False,True,0.0
686,Metro Market Nablus,Supermarkets,1748.0,2026-05-05 04:00:00,None,May,reel,Good morning ❤️😎 عنواننا || نابلس - رفيديا - ا...,125,0,...,0.0,681.0,Mixed,True,True,True,False,False,True,0.0


In [24]:
target_businesses = (
    df.loc[df['post_date'].isna(), 'business_name']
      .dropna()
      .unique()
)

print(target_businesses)

['LOFT Palestine' 'Balcon Boutique & Cafe' 'drift.for.men'
 'Fashion strawberry' 'Hijjawi Fashion - الحجاوي للأزياء' '528 Fashion'
 'Pardo Cafe' 'Hulk Gym / Diaa Shabaro' 'Metro Market Nablus']


In [25]:

for business in target_businesses:

    mask = df['business_name'] == business

    # Sort chronologically but preserve original index
    sub = df.loc[mask].copy().sort_values('post_date')

    # If all dates missing or only one valid date -> skip
    valid_dates = sub['post_date'].dropna()

    if len(valid_dates) < 2:
        continue

    # Compute average posting gap
    avg_gap = valid_dates.diff().dropna().mean()

    if pd.isna(avg_gap):
        avg_gap = pd.Timedelta(days=7)

    # Work on ordered indices
    ordered_idx = sub.index.tolist()

    for i, idx in enumerate(ordered_idx):

        if pd.isna(df.loc[idx, 'post_date']):

            # Search previous valid
            prev_date = None
            prev_i = None

            for j in range(i - 1, -1, -1):
                d = df.loc[ordered_idx[j], 'post_date']
                if pd.notna(d):
                    prev_date = d
                    prev_i = j
                    break

            # Search next valid
            next_date = None
            next_i = None

            for j in range(i + 1, len(ordered_idx)):
                d = df.loc[ordered_idx[j], 'post_date']
                if pd.notna(d):
                    next_date = d
                    next_i = j
                    break

            # CASE 1: Between two valid dates
            if prev_date is not None and next_date is not None:

                gap_steps = next_i - prev_i
                total_gap = next_date - prev_date

                interpolated = prev_date + total_gap * (
                    (i - prev_i) / gap_steps
                )

                df.loc[idx, 'post_date'] = interpolated

            # CASE 2: Missing at beginning
            elif prev_date is None and next_date is not None:

                distance = next_i - i
                interpolated = next_date - (avg_gap * distance)

                df.loc[idx, 'post_date'] = interpolated

            # CASE 3: Missing at end
            elif prev_date is not None and next_date is None:

                distance = i - prev_i
                interpolated = prev_date + (avg_gap * distance)

                df.loc[idx, 'post_date'] = interpolated

# Optional: convert back to date only
df['post_date'] = df['post_date'].dt.date

print(df[['business_name', 'post_date']].head())

    business_name   post_date
0  LOFT Palestine  2026-03-31
1  LOFT Palestine  2026-03-18
2  LOFT Palestine  2026-03-16
3  LOFT Palestine  2026-03-07
4  LOFT Palestine  2026-03-05


In [26]:
df.loc[:, 'post_date'] = pd.to_datetime(df['post_date'])

In [27]:
print("Remaining missing:", df['post_date'].isna().sum())

df[df['post_date'].isna()][['business_name', 'post_date']]

Remaining missing: 9


,business_name,post_date
64,Fashion strawberry,NaT
65,Fashion strawberry,NaT
66,Fashion strawberry,NaT
67,Fashion strawberry,NaT
68,Fashion strawberry,NaT
69,Fashion strawberry,NaT
70,Fashion strawberry,NaT
71,Fashion strawberry,NaT
72,Fashion strawberry,NaT


In [28]:
df['post_date'] = pd.to_datetime(df['post_date'], errors='coerce')

# remaining missing rows
remaining_mask = df['post_date'].isna()

# process per business
for business in df.loc[remaining_mask, 'business_name'].dropna().unique():

    business_mask = df['business_name'] == business
    missing_idx = df.loc[business_mask & df['post_date'].isna()].index

    # valid dates for this business
    valid_dates = (
        df.loc[business_mask, 'post_date']
        .dropna()
        .sort_values()
    )

    # fallback start date
    if len(valid_dates) == 0:
        start_date = pd.Timestamp('2026-01-01')

        # random average gap between 2–7 days
        avg_gap = pd.Timedelta(days=np.random.randint(2, 8))

    else:
        # infer average gap
        diffs = valid_dates.diff().dropna()

        if len(diffs) > 0:
            avg_gap = diffs.mean()
        else:
            avg_gap = pd.Timedelta(days=5)

        # avoid weird tiny/huge gaps
        avg_gap = pd.Timedelta(days=max(1, min(avg_gap.days, 14)))

        start_date = valid_dates.max() + avg_gap

    # generate sequential dates
    generated_dates = [
        (start_date + i * avg_gap).normalize()
        for i in range(len(missing_idx))
    ]

    # assign
    df.loc[missing_idx, 'post_date'] = generated_dates

# final check
print("Remaining missing:", df['post_date'].isna().sum())

Remaining missing: 0


`Date and time handled`

### `dailectal arabic`

In [29]:
df['arabic_dialect_style'].isna().sum()

np.int64(0)

In [30]:
levantine_mask = ~df['arabic_dialect_style'].isin([True, False])

In [31]:
df.loc[levantine_mask, 'arabic_dialect_style'] = True

In [32]:
df[~df['arabic_dialect_style'].isin([True, False])]

,business_name,sector,followers_count,post_date,day_of_week,month,post_type,caption_text,caption_length,hashtags_count,...,comments_count,views_count,language,CTA_present,promo_post,mentions_location,religious_theme,patriotic_theme,arabic_dialect_style,sponsored


`dialectal hadneld`

`Check for consistencies again`

In [33]:
print_inconsistencies()

sponsored: 2
sponsored: [0. 1.]
month: 12
month: ['March' 'February' None 'April' 'October' 'May' 'September' 'December'
 'August' 'June' 'January' 'July' 'November']
sector: 5
sector: ['Fashion' 'Cafes/Restaurants' 'Influencers' 'Gym' 'Supermarkets']
religious_theme: 2
religious_theme: [False  True]
patriotic_theme: 2
patriotic_theme: [False  True]
promo_post: 2
promo_post: [False  True]
day_of_week: 7
day_of_week: ['Tuesday' 'Wednesday' 'Monday' 'Saturday' 'Thursday' None 'Friday'
 'Sunday']
arabic_dialect_style: 2
arabic_dialect_style: [True False]
CTA_present: 2
CTA_present: [ True False]
post_type: 4
post_type: ['reel' 'post' 'Image' 'Reel']
language: 4
language: ['Arabic' 'English' 'No Caption' 'Mixed']
mentions_location: 2
mentions_location: [ True False]


In [34]:
print_missing()

followers_count    0.060563
day_of_week        0.074648
month              0.047887
likes_count        0.070423
comments_count     0.150704
views_count        0.281690
dtype: float64


### Fix missing months and day of week from the interpolated date 

In [35]:
df.loc[df['month'].isna(), 'month'] = df.loc[df['month'].isna(), 'post_date'].dt.month_name()

df.loc[df['day_of_week'].isna(), 'day_of_week'] = (
    df.loc[df['day_of_week'].isna(), 'post_date'].dt.day_name()
)

# Inspect remaining missing values
print("Missing month:", df['month'].isna().sum())
print("Missing day_of_week:", df['day_of_week'].isna().sum())

df[df['month'].isna() | df['day_of_week'].isna()][
    ['business_name', 'post_date', 'month', 'day_of_week']
]

Missing month: 0
Missing day_of_week: 0


,business_name,post_date,month,day_of_week


Month and Day week handled

In [36]:
print_missing()

followers_count    0.060563
likes_count        0.070423
comments_count     0.150704
views_count        0.281690
dtype: float64


In [37]:
print_inconsistencies()

sponsored: 2
sponsored: [0. 1.]
month: 12
month: ['March' 'February' 'April' 'May' 'October' 'June' 'January' 'September'
 'December' 'August' 'July' 'November']
sector: 5
sector: ['Fashion' 'Cafes/Restaurants' 'Influencers' 'Gym' 'Supermarkets']
religious_theme: 2
religious_theme: [False  True]
patriotic_theme: 2
patriotic_theme: [False  True]
promo_post: 2
promo_post: [False  True]
day_of_week: 7
day_of_week: ['Tuesday' 'Wednesday' 'Monday' 'Saturday' 'Thursday' 'Sunday' 'Friday']
arabic_dialect_style: 2
arabic_dialect_style: [True False]
CTA_present: 2
CTA_present: [ True False]
post_type: 4
post_type: ['reel' 'post' 'Image' 'Reel']
language: 4
language: ['Arabic' 'English' 'No Caption' 'Mixed']
mentions_location: 2
mentions_location: [ True False]


In [38]:
# Normalize post_type values

image_mask = df['post_type'] == 'Image'
reel_mask = df['post_type'] == 'Reel'

df.loc[image_mask, 'post_type'] = 'post'
df.loc[reel_mask, 'post_type'] = 'reel'

print(df['post_type'].unique())

['reel' 'post']



`Image and reel consistencies ahdnled`

In [39]:
print_missing()

followers_count    0.060563
likes_count        0.070423
comments_count     0.150704
views_count        0.281690
dtype: float64


### Propgate follower count from other posts or manually`

In [40]:
cols = [
    'followers_count',
    'likes_count',
    'comments_count',
    'views_count'
]

for col in cols:
    print(f"\n--- {col} ---")
    print(
        df[df[col].isna()]['business_name']
        .dropna()
        .unique()
    )


--- followers_count ---
['drift.for.men' 'Grand Shop' 'Noor Restaurant and Cafe'
 'Shawerma w Mashawi Al Helou' 'Hulk Gym / Diaa Shabaro']

--- likes_count ---
['LOFT Palestine' 'Balcon Boutique & Cafe' 'Grand Stores Mall'
 'KOTON Palestine' 'Fashion strawberry' 'Ghazala__store غزالة ستور'
 'Shamsboutique' 'جروان الدوار' 'Vienna Restaurant & Cafe'
 'Mazaj Oriental - مزاج الشرقي' 'KFC Palestine' 'Pardo Cafe'
 'Noor Restaurant and Cafe' 'Farouj Abu Alabd' 'Pulse Gym & Fitness'
 'Supermarket 25st']

--- comments_count ---
['LOFT Palestine' 'Balcon Boutique & Cafe' 'Fashion Shop Ramallah'
 'Grand Stores Mall' 'EUROMODA' 'Fashion strawberry'
 'Ghazala__store غزالة ستور' 'جروان الدوار' 'Pardo Cafe'
 'Noor Restaurant and Cafe' 'Farouj Abu Alabd' 'Hulk Gym / Diaa Shabaro'
 'Xtreme Fitness Ramallah' 'Jumbo Supermarket' 'Bravo Supermarket']

--- views_count ---
['LOFT Palestine' 'Balcon Boutique & Cafe' 'Grand Stores Mall'
 'Mikkawi 1955' 'Ghazala__store غزالة ستور' 'Lujain fashion'
 'Hijjawi F

In [41]:
def unique_business(col):
    businesses_missing_followers = (
        df.loc[df[col].isna(), 'business_name']
        .dropna()
        .unique()
    )

    for business in businesses_missing_followers:
        business_rows = df[df['business_name'] == business]
        available_followers = business_rows[col].dropna().unique()

        print(f"\nBusiness: {business}")
        print(f"Missing follower rows: {business_rows[col].isna().sum()}")

        if len(available_followers) > 0:
            print("Has other follower_count values:")
            print(available_followers)
        else:
            print("No follower_count available in other posts")

In [42]:
unique_business('followers_count')


Business: drift.for.men
Missing follower rows: 4
No follower_count available in other posts

Business: Grand Shop
Missing follower rows: 10
No follower_count available in other posts

Business: Noor Restaurant and Cafe
Missing follower rows: 12
Has other follower_count values:
[44000.]

Business: Shawerma w Mashawi Al Helou
Missing follower rows: 1
Has other follower_count values:
[81000.]

Business: Hulk Gym / Diaa Shabaro
Missing follower rows: 16
No follower_count available in other posts


In [43]:
# Businesses that have at least one missing follower_count
businesses_missing_followers = (
    df.loc[df['followers_count'].isna(), 'business_name']
      .dropna()
      .unique()
)

# Flag businesses where follower_count exists elsewhere
fillable_businesses = []

for business in businesses_missing_followers:

    business_mask = df['business_name'] == business

    available_followers = (
        df.loc[business_mask, 'followers_count']
          .dropna()
          .unique()
    )

    # If at least one valid follower count exists
    if len(available_followers) > 0:

        fillable_businesses.append(business)

        # Use the most common follower count
        fill_value = (
            df.loc[business_mask, 'followers_count']
              .mode()
              .iloc[0]
        )

        # Fill missing follower counts for this business
        df.loc[
            business_mask & df['followers_count'].isna(),
            'followers_count'
        ] = fill_value

        print(f"Filled {business} with follower_count = {fill_value}")

# Show businesses that were fillable
print("\nBusinesses filled:")
print(fillable_businesses)

# Remaining missing follower counts
print("\nRemaining missing follower_count:")
print(df['followers_count'].isna().sum())

Filled Noor Restaurant and Cafe with follower_count = 44000.0
Filled Shawerma w Mashawi Al Helou with follower_count = 81000.0

Businesses filled:
['Noor Restaurant and Cafe', 'Shawerma w Mashawi Al Helou']

Remaining missing follower_count:
30


In [44]:
df.loc[df['business_name'] == 'Hulk Gym / Diaa Shabaro', 'followers_count'] = 2404
df.loc[df['business_name'] == 'Grand Shop', 'followers_count'] = 45900
df.loc[df['business_name'] == 'drift.for.men', 'followers_count'] = 53500

In [45]:
print_missing()

likes_count       0.070423
comments_count    0.150704
views_count       0.281690
dtype: float64


`Propogated`

`Fill likes and comments with mean/median form the same page`

In [46]:
cols = ['likes_count', 'comments_count']

for col in cols:
    df[col] = df[col].fillna(
        df.groupby('business_name')[col].transform('median')
    )

In [47]:
print_missing()

likes_count       0.032394
comments_count    0.054930
views_count       0.281690
dtype: float64


In [48]:
# businesses where ALL posts are missing likes/comments

print("LIKES:")
print(
    df.groupby('business_name')['likes_count']
      .apply(lambda x: x.isna().all())
      .loc[lambda x: x]
)

print("\nCOMMENTS:")
print(
    df.groupby('business_name')['comments_count']
      .apply(lambda x: x.isna().all())
      .loc[lambda x: x]
)

LIKES:
business_name
Fashion strawberry    True
جروان الدوار          True
Name: likes_count, dtype: bool

COMMENTS:
business_name
Fashion Shop Ramallah    True
Fashion strawberry       True
Pardo Cafe               True
جروان الدوار             True
Name: comments_count, dtype: bool


In [49]:



# ---------- FILL LIKES WITH NOISE ----------

# Fashion strawberry -> around 20
mask1 = (
    (df['business_name'] == 'Fashion strawberry') &
    (df['likes_count'].isna())
)

df.loc[mask1, 'likes_count'] = np.random.randint(15, 26, mask1.sum())


# جروان الدوار -> around 30
mask2 = (
    (df['business_name'] == 'جروان الدوار') &
    (df['likes_count'].isna())
)

df.loc[mask2, 'likes_count'] = np.random.randint(25, 36, mask2.sum())


# ---------- REMAINING MISSING COMMENTS -> 0 ----------

df['comments_count'] = df['comments_count'].fillna(0)

In [50]:
print_missing()

views_count    0.28169
dtype: float64


`like and comment count handled`

### `View count`

In [51]:
df['views_count'] = df['views_count'].fillna(
    df.groupby('business_name')['views_count'].transform('median')
)

In [52]:
print_missing()

views_count    0.08169
dtype: float64


In [53]:
missing_view_businesses = (
    df.groupby('business_name')['views_count']
      .apply(lambda x: x.isna().all())
      .loc[lambda x: x]
      .index
)

df[df['business_name'].isin(missing_view_businesses)][
    ['business_name', 'post_type']
].drop_duplicates().sort_values('business_name')

,business_name,post_type
367,Abu shukri restaurant-مطعم حمص ابو شكري,post
162,Baby Bump,reel
549,FPAIspal,reel
551,Gintifada,reel
99,Hijjawi Fashion - الحجاوي للأزياء,reel
547,Hind Khoudary,reel
358,Jericho Resort Village - قرية أريحا السياحية,post
662,Katusha Supermarket,reel
79,Lujain fashion,reel
548,Moath Kahlout,reel


In [54]:
missing_view_businesses = (
    df.groupby('business_name')['views_count']
      .apply(lambda x: x.isna().all())
      .loc[lambda x: x]
      .index
)

summary = (
    df[df['business_name'].isin(missing_view_businesses)]
    .groupby('business_name')
    .agg(
        total_posts=('business_name', 'size'),
        posts_without_views=('views_count', lambda x: x.isna().sum()),
        post_types=('post_type', lambda x: list(x.dropna().unique()))
    )
    .reset_index()
)

print(summary)

                                   business_name  total_posts  \
0        Abu shukri restaurant-مطعم حمص ابو شكري            3   
1                                      Baby Bump           10   
2                                       FPAIspal            1   
3                                      Gintifada            1   
4              Hijjawi Fashion - الحجاوي للأزياء           12   
5                                  Hind Khoudary            1   
6   Jericho Resort Village - قرية أريحا السياحية            9   
7                            Katusha Supermarket            8   
8                                 Lujain fashion           11   
9                                  Moath Kahlout            1   
10                             The Tatreez Grove            1   

    posts_without_views post_types  
0                     3     [post]  
1                    10     [reel]  
2                     1     [reel]  
3                     1     [reel]  
4                    12     [reel]

In [55]:
mask1 = (
    (df['business_name'] == 'Hijjawi Fashion - الحجاوي للأزياء') &
    (df['views_count'].isna())
)

df.loc[mask1, 'views_count'] = np.random.randint(25000, 35001, mask1.sum())


# Lujain fashion - around 700
mask2 = (
    (df['business_name'] == 'Lujain fashion') &
    (df['views_count'].isna())
)

df.loc[mask2, 'views_count'] = np.random.randint(500, 901, mask2.sum())

In [56]:
print_missing()

views_count    0.049296
dtype: float64


`handled the above manually cuz they got a lot of missing based on their page median visual estimation and now we fill the rest using a hueristic`

In [57]:
# Estimate missing views from existing rows using median ratios:
# views_per_like and views_per_comment

valid_likes = df[
    df['views_count'].notna() &
    df['likes_count'].notna() &
    (df['likes_count'] > 0)
]

valid_comments = df[
    df['views_count'].notna() &
    df['comments_count'].notna() &
    (df['comments_count'] > 0)
]

views_per_like = (valid_likes['views_count'] / valid_likes['likes_count']).median()
views_per_comment = (valid_comments['views_count'] / valid_comments['comments_count']).median()

print("Median views per like:", views_per_like)
print("Median views per comment:", views_per_comment)


# Fill remaining missing views using likes/comments estimates
mask_missing_views = df['views_count'].isna()

estimate_from_likes = df['likes_count'] * views_per_like
estimate_from_comments = df['comments_count'] * views_per_comment

df.loc[mask_missing_views, 'views_count'] = (
    pd.concat(
        [estimate_from_likes[mask_missing_views], estimate_from_comments[mask_missing_views]],
        axis=1
    )
    .median(axis=1)
)

# optional: round to whole numbers
df['views_count'] = df['views_count'].round()

Median views per like: 151.23333333333335
Median views per comment: 3275.123482435489


In [58]:
print_missing()

Series([], dtype: float64)


In [60]:
print_inconsistencies()

sponsored: 2
sponsored: [0. 1.]
month: 12
month: ['March' 'February' 'April' 'May' 'October' 'June' 'January' 'September'
 'December' 'August' 'July' 'November']
sector: 5
sector: ['Fashion' 'Cafes/Restaurants' 'Influencers' 'Gym' 'Supermarkets']
religious_theme: 2
religious_theme: [False  True]
patriotic_theme: 2
patriotic_theme: [False  True]
promo_post: 2
promo_post: [False  True]
day_of_week: 7
day_of_week: ['Tuesday' 'Wednesday' 'Monday' 'Saturday' 'Thursday' 'Sunday' 'Friday']
arabic_dialect_style: 2
arabic_dialect_style: [True False]
CTA_present: 2
CTA_present: [ True False]
post_type: 2
post_type: ['reel' 'post']
language: 4
language: ['Arabic' 'English' 'No Caption' 'Mixed']
mentions_location: 2
mentions_location: [ True False]


In [61]:
# Save the processed DataFrame inside ../data
df.to_json(
    "../data/data_processed.json",
    orient="records",
    force_ascii=False,
    indent=4
)

print("Saved as ../data/data_processed.json")

Saved as ../data/data_processed.json
